# Report figure: training and inference performance

This notebook reads only saved timing summaries. It never retrains a
model and never repeats a benchmark. The exported figure is the single
report-ready comparison: training wall-clock time and `p95` model-only
inference time relative to the 26.2144 ms real-time budget.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import matplotlib.pyplot as plt

benchmark_dir = (
    WORK_ROOT
    / "outputs"
    / "performance_comparison"
    / "benchmark_runs"
    / "b0531_full_segments_model_only"
)
inference_path = benchmark_dir / "inference_timing_summary.csv"
training_path = benchmark_dir / "training_time_summary.csv"

if not inference_path.exists() or not training_path.exists():
    raise FileNotFoundError("Run 03_benchmark_model_inference.ipynb first.")

inference = pd.read_csv(inference_path)
training = pd.read_csv(training_path)

# The largest independent sample gives the most stable p95 estimate.
n_for_figure = int(inference["n_segments_in_estimate"].max())
inference = inference[inference["n_segments_in_estimate"] == n_for_figure].copy()
budget_ms = float(inference["budget_ms"].iloc[0])

order = [
    ("MLP (3 features)", "cpu", "MLP\n(3 features)"),
    ("1D CNN", "cpu", "1D CNN\nCPU"),
    ("1D CNN", "cuda", "1D CNN\nGPU"),
]
colors = ["#4C78A8", "#F58518", "#54A24B"]

def selected_rows(frame, value_column):
    values = []
    for model, device, label in order:
        row = frame[(frame["model"] == model) & (frame["device"] == device)]
        if len(row) != 1:
            raise ValueError(f"Expected exactly one row for {model} on {device}.")
        values.append((label, float(row.iloc[0][value_column])))
    return values

training_values = selected_rows(training, "training_wall_clock_s")
inference_values = selected_rows(inference, "p95_ms")


In [ ]:
fig, (ax_training, ax_inference) = plt.subplots(
    1,
    2,
    figsize=(10.6, 4.1),
    gridspec_kw={"width_ratios": [1, 1.15]},
)

x = np.arange(len(order))
labels = [label for _, _, label in order]

training_seconds = [value for _, value in training_values]
training_bars = ax_training.bar(x, training_seconds, color=colors, width=0.62)
ax_training.set_yscale("log")
ax_training.set_xticks(x, labels)
ax_training.set_ylabel("Wall-clock training time (s)")
ax_training.set_title("(a) Training")
ax_training.grid(axis="y", linestyle="--", alpha=0.35)

for bar, value in zip(training_bars, training_seconds):
    ax_training.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.1f}",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )

inference_ms = [value for _, value in inference_values]
inference_bars = ax_inference.bar(x, inference_ms, color=colors, width=0.62)
ax_inference.axhline(
    budget_ms,
    color="black",
    linestyle="--",
    linewidth=1.4,
    label=f"Real-time budget: {budget_ms:.2f} ms",
)
ax_inference.set_xticks(x, labels)
ax_inference.set_ylabel("Model inference time per segment, p95 (ms)")
ax_inference.set_title(f"(b) Inference ({n_for_figure} segments)")
ax_inference.grid(axis="y", linestyle="--", alpha=0.35)
ax_inference.legend(frameon=False, loc="upper right", fontsize=8.5)
ax_inference.set_ylim(0, max(max(inference_ms), budget_ms) * 1.22)

for bar, value in zip(inference_bars, inference_ms):
    ax_inference.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )

fig.suptitle("Performance of the evaluated ML models", y=1.02, fontsize=12)
fig.tight_layout()

figure_dir = benchmark_dir / "figures"
figure_dir.mkdir(exist_ok=True)
figure_base = figure_dir / "ml_model_training_and_inference_performance"
fig.savefig(figure_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
fig.savefig(figure_base.with_suffix(".pdf"), bbox_inches="tight")
plt.show()

print("Saved:", figure_base.with_suffix(".png"))
print("Saved:", figure_base.with_suffix(".pdf"))
